# A5 · Predicting frontier orbital energies with a neural network: what does the network add?
**Series A: Machine Learning Basics for Chemists, Post 5 of 7** &nbsp;|&nbsp; companion *primer* notebook

The full tutorial of this post (*Hammett Neural Networks*, Prof. V. S. Batista, CHEM 584) trains a small feedforward network, $10\to12\to8\to2$ with 254 parameters, on 30 tungsten benzylidyne complexes described by **Hammett substituent constants**, and predicts two DFT orbital quantities for 120 unseen complexes. Its own comparison shows that a plain **linear (Ridge) baseline is at least as good as the network**. This primer rebuilds the whole pipeline **from scratch with NumPy** (the data are embedded, nothing to download), reproduces the tutorial's baseline exactly, trains the network with hand-written backpropagation, and then asks *why* the network adds nothing here and what would change that.

> **Two parts.** **Part I** needs only NumPy, pandas and matplotlib and runs in about a minute on a CPU. **Part II** checks the implementation against scikit-learn and PyTorch.
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | the 150-complex library, the 30/120 split and the design of the training set |
| 2 | one neuron, the network, and its 254 parameters |
| 3 | train / validation / test, standardization, and leakage |
| 4 | the network from scratch: forward pass, backpropagation, a gradient check, Adam, learning curves |
| 5 | evaluation on 120 unseen complexes, against a Ridge baseline |
| 6 | thirty seeds, an ensemble, and is the across-seed spread an uncertainty? |
| 7 | why the baseline wins: training-set size, training-set design, the coefficients, screening |
| **Part II** | cross-check with scikit-learn and PyTorch |

## 0. Setup and data

In [ ]:
import io, math, time, importlib.util
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
mae = lambda p, y: np.abs(p - y).mean(0)
rmse = lambda p, y: np.sqrt(((p - y) ** 2).mean(0))
r2 = lambda p, y: 1 - ((p - y) ** 2).sum(0) / ((y - y.mean(0)) ** 2).sum(0)

In [ ]:
TRAIN_CSV = """
id,Rm,Rp,L1m,L1p,L2m,L2p,L3m,L3p,Xm,Xp,dxy,gap
111,0,0,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.34,0.06,-2.52,2.46529
122,0,0,0.1,0.13,0.1,0.13,0.1,0.13,0.37,0.23,-4.06,3.30474
133,0,0,0.22,0,0.22,0,0.22,0,0.12,-0.27,-3.95,3.42719
144,0,0,0.18,0.03,0.18,0.03,0.18,0.03,0.15,0,-4.12,3.34284
155,0,0,-0.23,-0.72,0.1,0.13,0.1,0.13,-0.16,-0.83,-3.48,2.84379
161,0,0,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.34,0.06,-3.36,2.98909
212,-0.07,-0.17,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.37,0.23,-2.57,2.34148
223,-0.07,-0.17,0.1,0.13,0.1,0.13,0.1,0.13,0.12,-0.27,-3.94,3.508
234,-0.07,-0.17,0.22,0,0.22,0,0.22,0,0.15,0,-3.93,3.14147
245,-0.07,-0.17,0.18,0.03,0.18,0.03,0.18,0.03,-0.16,-0.83,-3.79,3.27399
251,-0.07,-0.17,-0.23,-0.72,0.1,0.13,0.1,0.13,0.34,0.06,-3.5,3.06855
262,-0.07,-0.17,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.37,0.23,-3.37,2.87154
313,0.12,-0.27,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.12,-0.27,-2.39,2.67944
324,0.12,-0.27,0.1,0.13,0.1,0.13,0.1,0.13,0.15,0,-3.94,3.32433
335,0.12,-0.27,0.22,0,0.22,0,0.22,0,-0.16,-0.83,-3.83,3.33331
341,0.12,-0.27,0.18,0.03,0.18,0.03,0.18,0.03,0.34,0.06,-4.02,3.64759
352,0.12,-0.27,-0.23,-0.72,0.1,0.13,0.1,0.13,0.37,0.23,-3.49,3.05767
363,0.12,-0.27,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.12,-0.27,-3.2,3.13
414,0.15,0,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.15,0,-2.56,2.21794
425,0.15,0,0.1,0.13,0.1,0.13,0.1,0.13,-0.16,-0.83,-3.92,3.20743
431,0.15,0,0.22,0,0.22,0,0.22,0,0.34,0.06,-3.94,3.39616
442,0.15,0,0.18,0.03,0.18,0.03,0.18,0.03,0.37,0.23,-4.17929,3.42202
453,0.15,0,-0.23,-0.72,0.1,0.13,0.1,0.13,0.12,-0.27,-3.44,3.02011
464,0.15,0,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.15,0,-3.32,2.71835
515,-0.16,-0.83,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,-0.16,-0.83,-2.31,2.49331
521,-0.16,-0.83,0.1,0.13,0.1,0.13,0.1,0.13,0.34,0.06,-3.96,3.77711
532,-0.16,-0.83,0.22,0,0.22,0,0.22,0,0.37,0.23,-3.91,3.55073
543,-0.16,-0.83,0.18,0.03,0.18,0.03,0.18,0.03,0.12,-0.27,-3.86,3.68025
554,-0.16,-0.83,-0.23,-0.72,0.1,0.13,0.1,0.13,0.15,0,-3.4,2.99155
565,-0.16,-0.83,-0.23,-0.72,0.1,0.13,-0.23,-0.72,-0.16,-0.83,-3.15,3.01304
"""

TEST_CSV = """
id,Rm,Rp,L1m,L1p,L2m,L2p,L3m,L3p,Xm,Xp,dxy,gap
112,0,0,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.37,0.23,-2.60406,2.30584
113,0,0,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.12,-0.27,-2.4378,2.46582
114,0,0,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.15,0,-2.56161,2.17958
115,0,0,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,-0.16,-0.83,-2.47045,2.25413
121,0,0,0.1,0.13,0.1,0.13,0.1,0.13,0.34,0.06,-4.0484,3.46882
123,0,0,0.1,0.13,0.1,0.13,0.1,0.13,0.12,-0.27,-3.96541,3.46419
124,0,0,0.1,0.13,0.1,0.13,0.1,0.13,0.15,0,-3.99317,3.1676
125,0,0,0.1,0.13,0.1,0.13,0.1,0.13,-0.16,-0.83,-3.922,3.18759
131,0,0,0.22,0,0.22,0,0.22,0,0.34,0.06,-3.95072,3.38257
132,0,0,0.22,0,0.22,0,0.22,0,0.37,0.23,-4.0084,3.23807
134,0,0,0.22,0,0.22,0,0.22,0,0.15,0,-3.95915,3.10039
135,0,0,0.22,0,0.22,0,0.22,0,-0.16,-0.83,-3.88198,3.14757
141,0,0,0.18,0.03,0.18,0.03,0.18,0.03,0.34,0.06,-4.0767,3.45548
142,0,0,0.18,0.03,0.18,0.03,0.18,0.03,0.37,0.23,-4.20187,3.4114
143,0,0,0.18,0.03,0.18,0.03,0.18,0.03,0.12,-0.27,-3.9597,3.42257
145,0,0,0.18,0.03,0.18,0.03,0.18,0.03,-0.16,-0.83,-3.81276,3.23481
151,0,0,-0.23,-0.72,0.1,0.13,0.1,0.13,0.34,0.06,-3.5265,3.02773
152,0,0,-0.23,-0.72,0.1,0.13,0.1,0.13,0.37,0.23,-3.54228,2.87726
153,0,0,-0.23,-0.72,0.1,0.13,0.1,0.13,0.12,-0.27,-3.45276,3.01222
154,0,0,-0.23,-0.72,0.1,0.13,0.1,0.13,0.15,0,-3.51807,2.7393
162,0,0,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.37,0.23,-3.39507,2.82828
163,0,0,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.12,-0.27,-3.26555,2.97222
164,0,0,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.15,0,-3.34256,2.71808
165,0,0,-0.23,-0.72,0.1,0.13,-0.23,-0.72,-0.16,-0.83,-3.27725,2.79726
211,-0.07,-0.17,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.34,0.06,-2.48596,2.49657
213,-0.07,-0.17,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.12,-0.27,-2.40188,2.49249
214,-0.07,-0.17,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.15,0,-2.52732,2.21549
215,-0.07,-0.17,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,-0.16,-0.83,-2.43454,2.28597
221,-0.07,-0.17,0.1,0.13,0.1,0.13,0.1,0.13,0.34,0.06,-4.03425,3.51997
222,-0.07,-0.17,0.1,0.13,0.1,0.13,0.1,0.13,0.37,0.23,-4.04269,3.35699
224,-0.07,-0.17,0.1,0.13,0.1,0.13,0.1,0.13,0.15,0,-3.99072,3.194
225,-0.07,-0.17,0.1,0.13,0.1,0.13,0.1,0.13,-0.16,-0.83,-3.89858,3.23573
231,-0.07,-0.17,0.22,0,0.22,0,0.22,0,0.34,0.06,-3.93385,3.43018
232,-0.07,-0.17,0.22,0,0.22,0,0.22,0,0.37,0.23,-3.97929,3.28325
233,-0.07,-0.17,0.22,0,0.22,0,0.22,0,0.12,-0.27,-3.9284,3.46854
235,-0.07,-0.17,0.22,0,0.22,0,0.22,0,-0.16,-0.83,-3.849,3.19159
241,-0.07,-0.17,0.18,0.03,0.18,0.03,0.18,0.03,0.34,0.06,-4.05194,3.50066
242,-0.07,-0.17,0.18,0.03,0.18,0.03,0.18,0.03,0.37,0.23,-4.17793,3.46012
243,-0.07,-0.17,0.18,0.03,0.18,0.03,0.18,0.03,0.12,-0.27,-3.9382,3.46175
244,-0.07,-0.17,0.18,0.03,0.18,0.03,0.18,0.03,0.15,0,-4.08895,3.38665
252,-0.07,-0.17,-0.23,-0.72,0.1,0.13,0.1,0.13,0.37,0.23,-3.51317,2.92189
253,-0.07,-0.17,-0.23,-0.72,0.1,0.13,0.1,0.13,0.12,-0.27,-3.42637,3.04787
254,-0.07,-0.17,-0.23,-0.72,0.1,0.13,0.1,0.13,0.15,0,-3.49222,2.78148
255,-0.07,-0.17,-0.23,-0.72,0.1,0.13,0.1,0.13,-0.16,-0.83,-3.4563,2.88951
261,-0.07,-0.17,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.34,0.06,-3.33303,3.02501
263,-0.07,-0.17,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.12,-0.27,-3.23916,3.00461
264,-0.07,-0.17,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.15,0,-3.3178,2.75346
265,-0.07,-0.17,-0.23,-0.72,0.1,0.13,-0.23,-0.72,-0.16,-0.83,-3.25004,2.82501
311,0.12,-0.27,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.34,0.06,-2.45739,2.68324
312,0.12,-0.27,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.37,0.23,-2.5363,2.4729
314,0.12,-0.27,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.15,0,-2.4993,2.35046
315,0.12,-0.27,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,-0.16,-0.83,-2.39617,2.40759
321,0.12,-0.27,0.1,0.13,0.1,0.13,0.1,0.13,0.34,0.06,-4.01031,3.66719
322,0.12,-0.27,0.1,0.13,0.1,0.13,0.1,0.13,0.37,0.23,-4.02228,3.50038
323,0.12,-0.27,0.1,0.13,0.1,0.13,0.1,0.13,0.12,-0.27,-3.91371,3.65086
325,0.12,-0.27,0.1,0.13,0.1,0.13,0.1,0.13,-0.16,-0.83,-3.88688,3.38104
331,0.12,-0.27,0.22,0,0.22,0,0.22,0,0.34,0.06,-3.91834,3.58229
332,0.12,-0.27,0.22,0,0.22,0,0.22,0,0.37,0.23,-3.9616,3.43208
333,0.12,-0.27,0.22,0,0.22,0,0.22,0,0.12,-0.27,-3.89738,3.56133
334,0.12,-0.27,0.22,0,0.22,0,0.22,0,0.15,0,-3.92568,3.26882
342,0.12,-0.27,0.18,0.03,0.18,0.03,0.18,0.03,0.37,0.23,-4.145,3.59725
343,0.12,-0.27,0.18,0.03,0.18,0.03,0.18,0.03,0.12,-0.27,-3.91806,3.58038
344,0.12,-0.27,0.18,0.03,0.18,0.03,0.18,0.03,0.15,0,-4.05548,3.4427
345,0.12,-0.27,0.18,0.03,0.18,0.03,0.18,0.03,-0.16,-0.83,-3.76895,3.40569
351,0.12,-0.27,-0.23,-0.72,0.1,0.13,0.1,0.13,0.34,0.06,-3.47045,3.20488
353,0.12,-0.27,-0.23,-0.72,0.1,0.13,0.1,0.13,0.12,-0.27,-3.40351,3.17875
354,0.12,-0.27,-0.23,-0.72,0.1,0.13,0.1,0.13,0.15,0,-3.46664,2.90447
355,0.12,-0.27,-0.23,-0.72,0.1,0.13,0.1,0.13,-0.16,-0.83,-3.42555,3.01957
361,0.12,-0.27,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.34,0.06,-3.29303,3.14963
362,0.12,-0.27,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.37,0.23,-3.34038,3.00705
364,0.12,-0.27,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.15,0,-3.28052,2.86747
365,0.12,-0.27,-0.23,-0.72,0.1,0.13,-0.23,-0.72,-0.16,-0.83,-3.21167,2.93902
411,0.15,0,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.34,0.06,-2.51753,2.47128
412,0.15,0,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.37,0.23,-2.59263,2.3159
413,0.15,0,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.12,-0.27,-2.44678,2.47453
415,0.15,0,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,-0.16,-0.83,-2.45767,2.26176
421,0.15,0,0.1,0.13,0.1,0.13,0.1,0.13,0.34,0.06,-4.04459,3.48678
422,0.15,0,0.1,0.13,0.1,0.13,0.1,0.13,0.37,0.23,-4.05466,3.32351
423,0.15,0,0.1,0.13,0.1,0.13,0.1,0.13,0.12,-0.27,-3.95126,3.47236
424,0.15,0,0.1,0.13,0.1,0.13,0.1,0.13,0.15,0,-3.96949,3.16732
432,0.15,0,0.22,0,0.22,0,0.22,0,0.37,0.23,-3.98854,3.25222
433,0.15,0,0.22,0,0.22,0,0.22,0,0.12,-0.27,-3.92868,3.43971
434,0.15,0,0.22,0,0.22,0,0.22,0,0.15,0,-3.93194,3.10828
435,0.15,0,0.22,0,0.22,0,0.22,0,-0.16,-0.83,-3.856,3.15669
441,0.15,0,0.18,0.03,0.18,0.03,0.18,0.03,0.34,0.06,-4.058,3.46644
443,0.15,0,0.18,0.03,0.18,0.03,0.18,0.03,0.12,-0.27,-3.95262,3.41739
444,0.15,0,0.18,0.03,0.18,0.03,0.18,0.03,0.15,0,-4.09194,3.30855
445,0.15,0,0.18,0.03,0.18,0.03,0.18,0.03,-0.16,-0.83,-3.8,3.24708
451,0.15,0,-0.23,-0.72,0.1,0.13,0.1,0.13,0.34,0.06,-3.51045,3.03753
452,0.15,0,-0.23,-0.72,0.1,0.13,0.1,0.13,0.37,0.23,-3.52677,2.89086
454,0.15,0,-0.23,-0.72,0.1,0.13,0.1,0.13,0.15,0,-3.50337,2.75209
455,0.15,0,-0.23,-0.72,0.1,0.13,0.1,0.13,-0.16,-0.83,-3.46392,2.85821
461,0.15,0,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.34,0.06,-3.33603,2.98774
462,0.15,0,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.37,0.23,-3.38201,2.83807
463,0.15,0,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.12,-0.27,-3.2446,2.97141
465,0.15,0,-0.23,-0.72,0.1,0.13,-0.23,-0.72,-0.16,-0.83,-3.25494,2.78883
511,-0.16,-0.83,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.34,0.06,-2.36977,2.71835
512,-0.16,-0.83,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.37,0.23,-2.44596,2.56677
513,-0.16,-0.83,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.12,-0.27,-2.28733,2.70421
514,-0.16,-0.83,-0.23,-0.72,-0.23,-0.72,-0.23,-0.72,0.15,0,-2.42665,2.45032
522,-0.16,-0.83,0.1,0.13,0.1,0.13,0.1,0.13,0.37,0.23,-3.96106,3.6125
523,-0.16,-0.83,0.1,0.13,0.1,0.13,0.1,0.13,0.12,-0.27,-3.85575,3.7178
524,-0.16,-0.83,0.1,0.13,0.1,0.13,0.1,0.13,0.15,0,-3.90337,3.43127
525,-0.16,-0.83,0.1,0.13,0.1,0.13,0.1,0.13,-0.16,-0.83,-3.81613,3.48308
531,-0.16,-0.83,0.22,0,0.22,0,0.22,0,0.34,0.06,-3.86446,3.6944
533,-0.16,-0.83,0.22,0,0.22,0,0.22,0,0.12,-0.27,-3.84323,3.65929
534,-0.16,-0.83,0.22,0,0.22,0,0.22,0,0.15,0,-3.88786,3.38692
535,-0.16,-0.83,0.22,0,0.22,0,0.22,0,-0.16,-0.83,-3.77966,3.44742
541,-0.16,-0.83,0.18,0.03,0.18,0.03,0.18,0.03,0.34,0.06,-3.9616,3.75372
542,-0.16,-0.83,0.18,0.03,0.18,0.03,0.18,0.03,0.37,0.23,-4.08214,3.7099
544,-0.16,-0.83,0.18,0.03,0.18,0.03,0.18,0.03,0.15,0,-3.988,3.53576
545,-0.16,-0.83,0.18,0.03,0.18,0.03,0.18,0.03,-0.16,-0.83,-3.71327,3.4926
551,-0.16,-0.83,-0.23,-0.72,0.1,0.13,0.1,0.13,0.34,0.06,-3.40623,3.29358
552,-0.16,-0.83,-0.23,-0.72,0.1,0.13,0.1,0.13,0.37,0.23,-3.41467,3.15318
553,-0.16,-0.83,-0.23,-0.72,0.1,0.13,0.1,0.13,0.12,-0.27,-3.34201,3.25494
555,-0.16,-0.83,-0.23,-0.72,0.1,0.13,0.1,0.13,-0.16,-0.83,-3.36759,3.11263
561,-0.16,-0.83,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.34,0.06,-3.24133,3.2552
562,-0.16,-0.83,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.37,0.23,-3.2748,3.10719
563,-0.16,-0.83,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.12,-0.27,-3.15127,3.22065
564,-0.16,-0.83,-0.23,-0.72,0.1,0.13,-0.23,-0.72,0.15,0,-3.22283,2.9461
"""

In [ ]:
FEATURES = ["Rm", "Rp", "L1m", "L1p", "L2m", "L2p", "L3m", "L3p", "Xm", "Xp"]      # meta and para Hammett-style constants of R, L1, L2, L3, X
train = pd.read_csv(io.StringIO(TRAIN_CSV)); test = pd.read_csv(io.StringIO(TEST_CSV))
library = pd.concat([train, test], ignore_index=True)
library["R"], library["L"], library["X"] = library.id // 100, (library.id // 10) % 10, library.id % 10          # the three digits of the code
print("training set", train.shape, "| test set", test.shape, "| library", library.shape, "| columns:", list(library.columns[:13]))
print("targets: dxy orbital energy range", library.dxy.min().round(3), "to", library.dxy.max().round(3), "eV; dxy-LUMO gap range", library.gap.min().round(3), "to", library.gap.max().round(3), "eV")

## 1. The library, the split, and the design of the training set
The complexes are tungsten benzylidynes, aryl–C≡W, with a para substituent **R** on the ring, a ligand environment **L** (six choices; each of the three descriptor slots $L_1,L_2,L_3$ carries the constants of the donor groups) and a group **X** trans to the carbyne: $5\times6\times5=150$ complexes. A complex is named by a three-digit code `RLX`, for example `111`. Each substituent is described by **two numbers**, its Hammett constants $\sigma_{meta}$ and $\sigma_{para}$ (borrowed from organic chemistry and applied here to donor ligands as an approximation). The targets are two DFT quantities, the $d_{xy}$ orbital energy and the $d_{xy}$–LUMO gap, in eV.
**Stop and predict:** the training set has 30 of the 150 complexes. Is it a random sample, or designed?

In [ ]:
R_SIGMA = {1: ("H", 0.00, 0.00), 2: ("Me", -0.07, -0.17), 3: ("OMe", 0.12, -0.27), 4: ("SMe", 0.15, 0.00), 5: ("NMe2", -0.16, -0.83)}
X_SIGMA = {1: ("F", 0.34, 0.06), 2: ("Cl", 0.37, 0.23), 3: ("OMe", 0.12, -0.27), 4: ("SMe", 0.15, 0.00), 5: ("NMe2", -0.16, -0.83)}
ok_R = all((library.loc[library.R == k, ["Rm", "Rp"]].to_numpy() == v[1:]).all() for k, v in R_SIGMA.items())
ok_X = all((library.loc[library.X == k, ["Xm", "Xp"]].to_numpy() == v[1:]).all() for k, v in X_SIGMA.items())
print("every complex carries the (meta, para) constants of its R and X group:", ok_R and ok_X)
print(pd.DataFrame({"R index": list(R_SIGMA), "R": [v[0] for v in R_SIGMA.values()], "R (sigma_m, sigma_p)": [v[1:] for v in R_SIGMA.values()], "X": [v[0] for v in X_SIGMA.values()], "X (sigma_m, sigma_p)": [v[1:] for v in X_SIGMA.values()]}).to_string(index=False))
Ltab = library.groupby("L")[["L1m", "L1p", "L2m", "L2p", "L3m", "L3p"]].agg(lambda s: sorted(set(s))[0] if len(set(s)) == 1 else tuple(sorted(set(s))))
print("\nligand environments (the constants of the three descriptor slots L1, L2, L3):\n", Ltab.to_string())
print("\nall 150 descriptor vectors distinct:", len(library.drop_duplicates(FEATURES)) == 150)

d = train.assign(R=train.id // 100, L=(train.id // 10) % 10, X=train.id % 10)
print("\ntraining set design: (R, L) pairs covered:", d.groupby(["R", "L"]).ngroups, "of 30 | X values seen with each L:", d.groupby("L").X.nunique().tolist(), "| how often each X occurs:", d.X.value_counts().sort_index().tolist())
assert ok_R and ok_X and len(library.drop_duplicates(FEATURES)) == 150 and d.groupby(["R", "L"]).ngroups == 30 and d.X.value_counts().tolist() == [6] * 5
print("the 30 training complexes contain every (R, L) pair once, and X is balanced: a designed set, not a random one")

## 2. One neuron, and the network
A neuron computes $z=\sum_jw_jx_j+b$ and then an activation $a=f(z)$. The network of the tutorial has the **same 10 → 12 → 8 → 2 architecture** as the original Keras assignment: 12 $\tanh$ neurons, 8 ReLU neurons and 2 linear outputs, so $\mathbf h_1=\tanh(W_1\mathbf x+\mathbf b_1)$, $\mathbf h_2=\mathrm{ReLU}(W_2\mathbf h_1+\mathbf b_2)$, $\hat{\mathbf y}=W_3\mathbf h_2+\mathbf b_3$.
**Stop and predict:** for $x=(1,2)$, $w=(0.3,-0.4)$, $b=0.2$, the values of $z$, $\tanh z$ and ReLU$(z)$; and the number of trainable parameters.

In [ ]:
x_d, w_d, b_d = np.array([1.0, 2.0]), np.array([0.3, -0.4]), 0.2
z_d = w_d @ x_d + b_d; print(f"z = {z_d:.3f} | tanh(z) = {np.tanh(z_d):.5f} | ReLU(z) = {max(0.0, z_d):.3f}")
shapes = {"W1": (12, 10), "b1": (12,), "W2": (8, 12), "b2": (8,), "W3": (2, 8), "b3": (2,)}
n_par = sum(int(np.prod(s)) for s in shapes.values()); print("array shapes:", shapes); print("parameters: 120 + 12 + 96 + 8 + 16 + 2 =", n_par, "for 24 fitting molecules")
assert abs(z_d + 0.3) < 1e-12 and abs(np.tanh(z_d) + 0.291313) < 1e-6 and n_par == 254

## 3. Train, validation, test, and why the scaler is fitted on the fit set only
The 30 labelled complexes are split into **24 to fit** and **6 to validate**; the 120 other complexes are the **test set**, used once at the end. Every descriptor is standardized, $x'=(x-\mu)/s$, and so is every target, with $\mu,s$ computed **on the 24 fit molecules only**; the same numbers are then applied to validation and test. Fitting the scaler on all the data would let test information leak into training.

In [ ]:
Xtr, ytr = train[FEATURES].to_numpy(float), train[["dxy", "gap"]].to_numpy(float)
Xte, yte = test[FEATURES].to_numpy(float), test[["dxy", "gap"]].to_numpy(float)
perm = np.random.RandomState(42).permutation(30); ix_val, ix_fit = perm[:6], perm[6:]          # the same split as scikit-learn's train_test_split(..., test_size=0.2, random_state=42)
mu, sd = Xtr[ix_fit].mean(0), Xtr[ix_fit].std(0); ymu, ysd = ytr[ix_fit].mean(0), ytr[ix_fit].std(0)
Z = lambda X: (X - mu) / sd
Xf, Xv, Xt = Z(Xtr[ix_fit]), Z(Xtr[ix_val]), Z(Xte); Yf, Yv = (ytr[ix_fit] - ymu) / ysd, (ytr[ix_val] - ymu) / ysd
print("fit", Xf.shape, "validation", Xv.shape, "test", Xt.shape, "| first descriptor: mean", round(mu[0], 5), "scale", round(sd[0], 8), "| target means (eV):", ymu.round(8))
print("mean of the standardized fit descriptors:", Xf.mean(0).round(12)[:3], "...  the test descriptors are NOT centred:", Xt.mean(0).round(3)[:3], "...")
assert abs(mu[0] - 0.00625) < 1e-9 and abs(sd[0] - 0.11488445282108455) < 1e-9 and np.allclose(ymu, [-3.47038708, 3.06861083], atol=1e-7)

## 4. The network from scratch
**Forward pass** as in Section 2; **loss** $L=\frac1{NK}\sum_{i,k}(\hat y_{ik}-y_{ik})^2$ on the standardized targets; **backpropagation** by the chain rule, layer by layer ($\partial L/\partial\hat y=2(\hat y-y)/(NK)$, then through the weights, through ReLU's $\mathbf 1[z>0]$ and through $\tanh'=1-\tanh^2$); **Adam** with an L2 weight decay of $0.01$ added to the gradient, full batch, 350 epochs; the weights of the epoch with the lowest validation loss are kept. The initial weights follow PyTorch's default, uniform in $\pm1/\sqrt{\text{fan-in}}$.
**Stop and predict:** which will be lower at the end, the training or the validation loss? Will a smaller learning rate help?

In [ ]:
def init(seed):
    r = np.random.default_rng(seed); u = lambda fo, fi: r.uniform(-1 / np.sqrt(fi), 1 / np.sqrt(fi), (fo, fi)); ub = lambda fo, fi: r.uniform(-1 / np.sqrt(fi), 1 / np.sqrt(fi), fo)
    return {"W1": u(12, 10), "b1": ub(12, 10), "W2": u(8, 12), "b2": ub(8, 12), "W3": u(2, 8), "b3": ub(2, 8)}
def forward(p, X):
    z1 = X @ p["W1"].T + p["b1"]; h1 = np.tanh(z1); z2 = h1 @ p["W2"].T + p["b2"]; h2 = np.maximum(z2, 0); return z1, h1, z2, h2, h2 @ p["W3"].T + p["b3"]
def grads(p, X, Y):
    z1, h1, z2, h2, yh = forward(p, X); N, K = Y.shape; d3 = 2 * (yh - Y) / (N * K)
    g = {"W3": d3.T @ h2, "b3": d3.sum(0)}; d2 = (d3 @ p["W3"]) * (z2 > 0); g["W2"] = d2.T @ h1; g["b2"] = d2.sum(0)
    d1 = (d2 @ p["W2"]) * (1 - h1 ** 2); g["W1"] = d1.T @ X; g["b1"] = d1.sum(0); return g, ((yh - Y) ** 2).mean()

# gradient check against finite differences, on every one of the 254 parameters
p0 = init(1); g0, _ = grads(p0, Xf, Yf); worst = 0.0
for k in p0:
    for idx in np.ndindex(*p0[k].shape):
        up = {a: b.copy() for a, b in p0.items()}; dn = {a: b.copy() for a, b in p0.items()}; up[k][idx] += 1e-6; dn[k][idx] -= 1e-6
        fd = (grads(up, Xf, Yf)[1] - grads(dn, Xf, Yf)[1]) / 2e-6; worst = max(worst, abs(fd - g0[k][idx]))
print(f"largest |finite difference - backpropagation| over all {n_par} parameters: {worst:.1e}"); assert worst < 1e-7

def train_net(Xa, Ya, Xb, Yb, seed=42, epochs=350, lr=0.01, wd=0.01):
    p = init(seed); m = {k: np.zeros_like(v) for k, v in p.items()}; v = {k: np.zeros_like(x) for k, x in p.items()}; best = (np.inf, None, 0); hist = []
    for t in range(1, epochs + 1):
        g, l = grads(p, Xa, Ya)
        for k in p:
            gk = g[k] + wd * p[k]; m[k] = 0.9 * m[k] + 0.1 * gk; v[k] = 0.999 * v[k] + 0.001 * gk ** 2
            p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
        vl = ((forward(p, Xb)[4] - Yb) ** 2).mean(); hist.append((l, vl))
        if vl < best[0]: best = (vl, {k: x.copy() for k, x in p.items()}, t)
    return best[1], best[2], np.array(hist)
t0 = time.perf_counter(); net, ep, hist = train_net(Xf, Yf, Xv, Yv); print(f"selected epoch (lowest validation loss): {ep} | minimum validation MSE {hist[:, 1].min():.4f} | final training MSE {hist[-1, 0]:.4f} | {time.perf_counter() - t0:.2f} s")
rows = []
for lr in (0.01, 0.001):
    _, e_, h_ = train_net(Xf, Yf, Xv, Yv, lr=lr); rows.append((lr, e_, h_[:, 1].min(), h_[-1, 0]))
print(pd.DataFrame(rows, columns=["learning rate", "selected epoch", "min validation MSE", "final training MSE"]).round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.8)); e_ax = np.arange(1, 351)
ax.semilogy(e_ax, hist[:, 0], color=COLORS["blue"], label="training MSE"); ax.semilogy(e_ax, hist[:, 1], color=COLORS["orange"], label="validation MSE"); ax.axvline(ep, color="k", ls="--", lw=1, label=f"selected epoch {ep}")
ax.set(xlabel="epoch", ylabel="MSE (standardized units)", title="Learning curves of the NumPy network"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 5. Evaluation on 120 unseen complexes, against a Ridge baseline
**MAE** is the mean absolute error, **RMSE** weights large errors more, $R^2=1-\sum r_i^2/\sum(y_i-\bar y)^2$. A neural network must be compared with a **simple baseline** trained on the *same* data: Ridge regression, $\hat y=\beta_0+\sum_j\beta_jx_j$ with an L2 penalty ($\alpha=1$), which has a closed-form solution and no random initialization.
**Stop and predict:** with only 24 fitting molecules and a network of 254 parameters, which will have the lower test error?

In [ ]:
def ridge_fit(X, y, alpha=1.0):
    xm, ym = X.mean(0), y.mean(0); Xc = X - xm; W = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(X.shape[1]), Xc.T @ (y - ym)); return W, ym - xm @ W
W_r, b_r = ridge_fit(Xf, ytr[ix_fit]); pred_ridge = Xt @ W_r + b_r                           # trained on the raw targets in eV
pred_nn = forward(net, Xt)[4] * ysd + ymu
tab = pd.DataFrame([(m, t, a, b, c) for m, pr in (("neural network (NumPy)", pred_nn), ("Ridge baseline", pred_ridge)) for t, a, b, c in zip(("dxy", "gap"), mae(pr, yte), rmse(pr, yte), r2(pr, yte))],
                   columns=["model", "target", "MAE (eV)", "RMSE (eV)", "R2"]); print(tab.round(4).to_string(index=False))
print("\nthe tutorial's table for its PyTorch network: dxy MAE 0.0486, RMSE 0.0593, R2 0.9881 | gap MAE 0.0931, RMSE 0.1129, R2 0.9183")
print("the tutorial's table for Ridge:               dxy MAE 0.0423, RMSE 0.0542, R2 0.9901 | gap MAE 0.0885, RMSE 0.1039, R2 0.9309  (reproduced exactly above)")
assert np.allclose(mae(pred_ridge, yte), [0.04233, 0.08848], atol=2e-4) and np.allclose(rmse(pred_ridge, yte), [0.0542, 0.1039], atol=2e-4) and np.allclose(r2(pred_ridge, yte), [0.99009, 0.93086], atol=2e-4)
for j, t in enumerate(("dxy", "gap")):
    err = np.abs(pred_nn[:, j] - yte[:, j]); w = np.argsort(-err)[:5]
    print(f"\nfive largest errors of the network, {t}:", [(int(test.id.iloc[i]), round(float(err[i]), 3)) for i in w])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for j, (a, t) in enumerate(zip(ax, ("dxy orbital energy (eV)", "dxy-LUMO gap (eV)"))):
    a.scatter(yte[:, j], pred_nn[:, j], s=14, color=COLORS["blue"], alpha=0.75, label="neural network"); a.scatter(yte[:, j], pred_ridge[:, j], s=14, color=COLORS["orange"], alpha=0.6, label="Ridge")
    lo, hi = yte[:, j].min(), yte[:, j].max(); a.plot([lo, hi], [lo, hi], "k--", lw=1); a.set(xlabel="reference (eV)", ylabel="predicted (eV)", title=t)
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

## 6. Thirty seeds, an ensemble, and the meaning of the spread
One seed is one random initialization. We repeat the training for **30 seeds** (same split), compare every network with the Ridge baseline, form the **ensemble mean** (the tutorial uses three seeds and an ensemble for screening), and ask whether the **across-seed standard deviation** is a usable uncertainty: if it were calibrated, about 68 % of the absolute errors would be below one standard deviation.
**Stop and predict:** how many of 30 networks will beat Ridge? Will the standard deviation be calibrated?

In [ ]:
P, res = [], []
for s in range(30):
    p_, e_, h_ = train_net(Xf, Yf, Xv, Yv, seed=s); pr_ = forward(p_, Xt)[4] * ysd + ymu; P.append(pr_); res.append(np.r_[mae(pr_, yte), rmse(pr_, yte)])
P, res = np.array(P), np.array(res); mae_ridge = mae(pred_ridge, yte)
print(pd.DataFrame({"": ["NN, mean of 30 seeds", "NN, spread (sd)", "NN, best seed", "NN, worst seed", "Ridge"], "dxy MAE (eV)": [res[:, 0].mean(), res[:, 0].std(ddof=1), res[:, 0].min(), res[:, 0].max(), mae_ridge[0]],
                    "gap MAE (eV)": [res[:, 1].mean(), res[:, 1].std(ddof=1), res[:, 1].min(), res[:, 1].max(), mae_ridge[1]]}).round(4).to_string(index=False))
beat = ((res[:, 0] < mae_ridge[0]).sum(), (res[:, 1] < mae_ridge[1]).sum()); print(f"\nnetworks with a test MAE below Ridge's: dxy {beat[0]} of 30, gap {beat[1]} of 30")
mean_p, sd_p = P.mean(0), P.std(0, ddof=1); ens = mae(mean_p, yte); print("ensemble of 30 (average of the predictions):", ens.round(4), "| a single network on average:", res[:, :2].mean(0).round(4))
for j, t in enumerate(("dxy", "gap")):
    err = np.abs(mean_p[:, j] - yte[:, j]); inside = 100 * np.mean(err <= sd_p[:, j]); sp = pd.Series(sd_p[:, j]).corr(pd.Series(err), method="spearman")
    print(f"{t}: median across-seed sd {np.median(sd_p[:, j]):.4f} eV, median |error| {np.median(err):.4f} eV | errors inside one sd: {inside:.0f} % (68 % if calibrated) | Spearman(sd, |error|) {sp:+.2f}")
assert beat[0] <= 3 and beat[1] <= 3 and 0.045 < res[:, 0].mean() < 0.057 and 100 * np.mean(np.abs(mean_p[:, 0] - yte[:, 0]) <= sd_p[:, 0]) < 40

## 7. Why the baseline wins, and what would change it
Five questions. **(a) More data:** train both models on random subsets of growing size and test on 50 other complexes. **(b) Design:** is the tutorial's designed training set better than a random 30? **(c) The coefficients:** what does the linear model use? **(d) Screening:** which model finds the best complexes? **(e) Encoding:** would a different description of the same complexes do better than the Hammett constants?
**Stop and predict:** how many training complexes does the network need to catch up with Ridge? And is the problem the model or the descriptors?

In [ ]:
A = library[FEATURES].to_numpy(float); Y = library[["dxy", "gap"]].to_numpy(float); ids = library.id.to_numpy()
def run_models(tr_i, te_i, seed=0, use_nn=True):
    m_, s_ = A[tr_i].mean(0), A[tr_i].std(0) + 1e-12; W, b = ridge_fit((A[tr_i] - m_) / s_, Y[tr_i]); out = {"ridge": mae(((A[te_i] - m_) / s_) @ W + b, Y[te_i])}
    if use_nn:
        pm = np.random.RandomState(seed).permutation(len(tr_i)); nv = max(2, int(round(0.2 * len(tr_i)))); v_i, f_i = tr_i[pm[:nv]], tr_i[pm[nv:]]
        m2, s2 = A[f_i].mean(0), A[f_i].std(0) + 1e-12; ym_, ys_ = Y[f_i].mean(0), Y[f_i].std(0)
        p_, _, _ = train_net((A[f_i] - m2) / s2, (Y[f_i] - ym_) / ys_, (A[v_i] - m2) / s2, (Y[v_i] - ym_) / ys_, seed=seed); out["nn"] = mae(forward(p_, (A[te_i] - m2) / s2)[4] * ys_ + ym_, Y[te_i])
    return out
rng = np.random.default_rng(1); rows = []
for n_tr in (10, 20, 30, 50, 75, 100):
    r_, n_ = [], []
    for rep in range(15):
        pm = rng.permutation(150); o = run_models(pm[50:50 + n_tr], pm[:50], seed=rep); r_.append(o["ridge"]); n_.append(o["nn"])
    r_, n_ = np.array(r_).mean(0), np.array(n_).mean(0); rows.append((n_tr, r_[0], n_[0], r_[1], n_[1]))
lc = pd.DataFrame(rows, columns=["training complexes", "Ridge dxy", "NN dxy", "Ridge gap", "NN gap"]); print("(a) mean test MAE (eV) over 15 random splits, tested on 50 other complexes:\n", lc.round(4).to_string(index=False))
folds = np.array_split(np.random.default_rng(3).permutation(150), 5); cv = []
for k in range(5):
    o = run_models(np.concatenate([folds[j] for j in range(5) if j != k]), folds[k], seed=k); cv.append(np.r_[o["ridge"], o["nn"]])
cv = np.array(cv).mean(0); print(f"    5-fold cross-validation on all 150 (120 training complexes each): Ridge MAE {cv[:2].round(4)} | network MAE {cv[2:].round(4)}")
assert lc.iloc[0]["Ridge dxy"] < lc.iloc[0]["NN dxy"] and lc.iloc[2]["Ridge dxy"] < lc.iloc[2]["NN dxy"] and lc.iloc[2]["Ridge gap"] < lc.iloc[2]["NN gap"] and lc.iloc[-1]["NN dxy"] < 1.15 * lc.iloc[-1]["Ridge dxy"]

In [ ]:
d_idx = np.array([int(np.where(ids == i)[0][0]) for i in train.id]); rest = np.setdiff1d(np.arange(150), d_idx)
des = run_models(d_idx, rest, use_nn=False)["ridge"]; rg = np.random.default_rng(7); rnd = []
for _ in range(500):
    pm = rg.permutation(150); rnd.append(run_models(pm[:30], pm[30:], use_nn=False)["ridge"])
rnd = np.array(rnd)
print("(b) Ridge test MAE (eV), trained on 30 complexes and tested on the other 120:")
print(f"    the designed training set: dxy {des[0]:.4f}, gap {des[1]:.4f}")
print(f"    500 random 30-subsets:     dxy {rnd[:, 0].mean():.4f} +- {rnd[:, 0].std():.4f}, gap {rnd[:, 1].mean():.4f} +- {rnd[:, 1].std():.4f}")
share = (100 * (rnd[:, 0] > des[0]).mean(), 100 * (rnd[:, 1] > des[1]).mean()); print(f"    random subsets with a LARGER error than the designed one: dxy {share[0]:.0f} %, gap {share[1]:.0f} %")
assert min(share) > 90

In [ ]:
m30, s30 = A[d_idx].mean(0), A[d_idx].std(0); W30, _ = ridge_fit((A[d_idx] - m30) / s30, Y[d_idx])
print("(c) standardized Ridge coefficients on the 30 designed complexes (eV per standard deviation of the descriptor):"); print(pd.DataFrame(W30, index=FEATURES, columns=["dxy", "gap"]).round(3).to_string())
C = pd.DataFrame(A[d_idx], columns=FEATURES).corr().abs().to_numpy().copy(); np.fill_diagonal(C, 0)
pairs = sorted({(FEATURES[a], FEATURES[b], round(float(C[a, b]), 3)) for a in range(10) for b in range(a + 1, 10)}, key=lambda t: -t[2])[:4]
print("\nmeta and para constants of the same group are strongly correlated:", pairs, "-> read the coefficients of a group together, not one by one")
assert pairs[0][2] > 0.9
assert (W30[2:8, 0] < 0).all()                                   # the six ligand coefficients for dxy
print("\nsign of the effect on dxy: all the ligand coefficients are negative, i.e. more electron-withdrawing constants (larger sigma) lower the dxy energy")

In [ ]:
top_true = set(np.argsort(-yte[:, 0])[:10]); ranks = {}
for name, pp in (("Ridge", pred_ridge), ("network, seed 42", pred_nn), ("ensemble of 30 networks", mean_p)):
    hit = len(top_true & set(np.argsort(-pp[:, 0])[:10])); sp = pd.Series(pp[:, 0]).corr(pd.Series(yte[:, 0]), method="spearman"); ranks[name] = hit
    print(f"(d) {name:24s}: {hit} of the true top-10 (highest dxy energy) are in its predicted top-10 | Spearman rank correlation {sp:.3f}")
print("true top-5 complexes:", test.id.to_numpy()[np.argsort(-yte[:, 0])[:5]].tolist(), "| Ridge top-5:", test.id.to_numpy()[np.argsort(-pred_ridge[:, 0])[:5]].tolist(), "| ensemble top-5:", test.id.to_numpy()[np.argsort(-mean_p[:, 0])[:5]].tolist())
assert ranks["Ridge"] >= 6 and ranks["ensemble of 30 networks"] >= 6

**(e) A different encoding.** The Hammett constants compress each group to two numbers. The library is a full factorial of three *categorical* choices, R, L and X. How much does each choice explain by itself, and how does an **additive model with one parameter per group** (4 + 5 + 4 differences from a reference group, plus an intercept: 14 parameters) do, trained on the *same* designed 30 complexes and tested on the *same* 120? Such a model is a lookup table: it cannot predict a group that is not in the library, which the Hammett encoding could in principle.

In [ ]:
def eta2(col, tgt):
    m_ = library.groupby(col)[tgt].transform("mean"); return 1 - ((library[tgt] - m_) ** 2).sum() / ((library[tgt] - library[tgt].mean()) ** 2).sum()
print("(e) fraction of the variance explained by one factor alone:\n", pd.DataFrame({t: {f: round(eta2(f, t), 3) for f in ("R", "L", "X")} for t in ("dxy", "gap")}).T.to_string())
def onehot(df):
    cols = [np.ones(len(df))]
    for f, levels in (("R", range(2, 6)), ("L", range(2, 7)), ("X", range(2, 6))): cols += [(df[f] == k).astype(float).to_numpy() for k in levels]
    return np.column_stack(cols)
Dall = onehot(library); trn_i, tst_i = np.arange(30), np.arange(30, 150)               # the first 30 rows of the library are the designed training set
beta = np.linalg.lstsq(Dall[trn_i], Y[trn_i], rcond=None)[0]; pred_add = Dall[tst_i] @ beta
print(f"\nadditive model, {Dall.shape[1]} parameters, trained on the designed 30 and tested on the 120: MAE dxy {mae(pred_add, Y[tst_i])[0]:.4f}, gap {mae(pred_add, Y[tst_i])[1]:.4f} eV | R2 {r2(pred_add, Y[tst_i]).round(4)}")
print(f"Ridge on the Hammett descriptors, same data:                          MAE dxy {mae(pred_ridge, yte)[0]:.4f}, gap {mae(pred_ridge, yte)[1]:.4f} eV")
rg = np.random.default_rng(5); out = []
for _ in range(300):
    pm = rg.permutation(150); t_, e_ = pm[:30], pm[30:]
    if np.linalg.matrix_rank(Dall[t_]) < Dall.shape[1]: continue                      # a random 30-subset may not identify all 14 parameters
    b_ = np.linalg.lstsq(Dall[t_], Y[t_], rcond=None)[0]; out.append(np.r_[mae(Dall[e_] @ b_, Y[e_]), run_models(t_, e_, use_nn=False)["ridge"]])
out = np.array(out); better = (100 * np.mean(out[:, 0] < out[:, 2]), 100 * np.mean(out[:, 1] < out[:, 3]))
print(f"{len(out)} random 30-subsets with an identifiable design: additive MAE {out[:, 0].mean():.4f} / {out[:, 1].mean():.4f} against Ridge {out[:, 2].mean():.4f} / {out[:, 3].mean():.4f}; the additive model is better in {better[0]:.0f} % (dxy) and {better[1]:.0f} % (gap) of them")
Zs = lambda X_: (X_ - A[:30].mean(0)) / A[:30].std(0); W_, b0_ = ridge_fit(Zs(A[:30]), Y[:30]); resid = Y[tst_i] - (Zs(A[tst_i]) @ W_ + b0_)
by_L = pd.DataFrame({"L": library.L.iloc[tst_i].to_numpy(), "gap": resid[:, 1], "dxy": resid[:, 0]}).groupby("L").mean()
print("\nmean test residual (reference minus Ridge prediction, eV) by ligand environment:\n", by_L.T.round(3).to_string())
means = library.groupby("L")[["dxy", "gap"]].mean(); print("\nmean reference values of ligand environments 3 and 4 (eV):", means.loc[[3, 4]].round(3).to_dict("index"), "; their constants (0.22, 0.00) and (0.18, 0.03) are nearly the same")
assert mae(pred_add, Y[tst_i])[1] < 0.04 and mae(pred_add, Y[tst_i])[1] < 0.5 * mae(pred_ridge, yte)[1] and better[1] > 95 and eta2("L", "dxy") > 0.95

## 8. Your turn
**Exercise 1.** Change the weight decay to $0$ and to $0.1$. Which gives the lowest validation loss, and what happens to the test error?
**Exercise 2.** Replace the tanh/ReLU layers by a single linear layer (a network with no hidden layer). What model do you obtain?
**Exercise 3.** Use the gap as the only target. Does the network do better than for the two-target network?
**Exercise 4.** Train on the 120 complexes of the test file and test on the designed 30. Which model wins now?
**Exercise 5.** Add the product $\sigma_{X,para}\,\sigma_{R,para}$ as an eleventh descriptor to Ridge. Does it improve the gap?
**Exercise 6.** Give the *network* the one-hot encoding of Section 7(e) (14 inputs) instead of the ten Hammett constants. Does it now beat the additive model?

## 9. Self-check

In [ ]:
assert worst < 1e-7 and n_par == 254 and ok_R and ok_X
assert np.allclose(mae(pred_ridge, yte), [0.04233, 0.08848], atol=2e-4) and beat[0] <= 3 and beat[1] <= 3
assert lc.iloc[2]["Ridge dxy"] < lc.iloc[2]["NN dxy"] and min(share) > 90 and ranks["Ridge"] >= 6
assert mae(pred_add, Y[tst_i])[1] < 0.04 and better[1] > 95
print("All checks passed: you reproduced the numbers in the post.")

# Part II. Cross-check with scikit-learn and PyTorch
> **Read this first.** Part II checks the NumPy implementation of Part I against the libraries the full tutorial uses: **scikit-learn** (the split, the scaler and Ridge) and **PyTorch** (the same 10→12→8→2 network: forward pass, loss and gradients for identical weights; then the tutorial's own training procedure). Colab has both preinstalled. Each stage is guarded: if a package is missing or its interface differs, the cell prints what failed and the others still run. The last cell lists the checks that passed.

In [ ]:
HAVE_SK = importlib.util.find_spec("sklearn") is not None; HAVE_TORCH = importlib.util.find_spec("torch") is not None
checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
if HAVE_SK:
    try:
        from sklearn.model_selection import train_test_split
        from sklearn.preprocessing import StandardScaler
        from sklearn.linear_model import Ridge
        sk_fit, sk_val = train_test_split(np.arange(30), test_size=0.2, random_state=42)
        check("our NumPy permutation reproduces train_test_split(test_size=0.2, random_state=42)", np.array_equal(sk_fit, ix_fit) and np.array_equal(sk_val, ix_val))
        sc = StandardScaler().fit(Xtr[ix_fit]); check("the standardization equals StandardScaler (mean and scale)", np.allclose(sc.mean_, mu) and np.allclose(sc.scale_, sd))
        rd = Ridge(alpha=1.0).fit(sc.transform(Xtr[ix_fit]), ytr[ix_fit]); pr_sk = rd.predict(sc.transform(Xte))
        check(f"our closed-form Ridge equals sklearn's Ridge (largest prediction difference {np.abs(pr_sk - pred_ridge).max():.1e} eV)", np.abs(pr_sk - pred_ridge).max() < 1e-8)
    except Exception as e: print("scikit-learn stage could not be completed:", type(e).__name__, e)
else: print("scikit-learn is not available: install it (pip install scikit-learn) and re-run this cell.")

In [ ]:
if HAVE_TORCH:
    try:
        import torch
        from torch import nn
        torch.manual_seed(7)
        net_t = nn.Sequential(nn.Linear(10, 12), nn.Tanh(), nn.Linear(12, 8), nn.ReLU(), nn.Linear(8, 2))
        check(f"the PyTorch network has {sum(p.numel() for p in net_t.parameters())} parameters (Part I: {n_par})", sum(p.numel() for p in net_t.parameters()) == n_par)
        pw = {"W1": net_t[0].weight, "b1": net_t[0].bias, "W2": net_t[2].weight, "b2": net_t[2].bias, "W3": net_t[4].weight, "b3": net_t[4].bias}
        p_np = {k: v.detach().numpy().astype(float) for k, v in pw.items()}                               # the same weights, in our NumPy network
        xb, yb = torch.tensor(Xf, dtype=torch.float32), torch.tensor(Yf, dtype=torch.float32)
        out_t = net_t(xb); loss_t = nn.MSELoss()(out_t, yb); loss_t.backward()
        g_np, loss_np = grads(p_np, Xf.astype(np.float32).astype(float), Yf.astype(np.float32).astype(float))
        check(f"forward pass equals PyTorch (largest difference {np.abs(out_t.detach().numpy() - forward(p_np, Xf)[4]).max():.1e})", np.abs(out_t.detach().numpy() - forward(p_np, Xf)[4]).max() < 1e-5)
        check(f"MSE loss equals PyTorch ({loss_t.item():.6f} against {loss_np:.6f})", abs(loss_t.item() - loss_np) < 1e-6)
        gd = max(np.abs(pw[k].grad.numpy() - g_np[k]).max() for k in pw); check(f"backpropagation equals PyTorch autograd for all six arrays (largest difference {gd:.1e})", gd < 1e-6)
    except Exception as e: print("PyTorch comparison could not be completed:", type(e).__name__, e)
else: print("PyTorch is not available: install it (pip install torch) and re-run this cell.")

In [ ]:
if HAVE_TORCH:
    try:
        def make_model(seed=42):
            torch.manual_seed(seed); return nn.Sequential(nn.Linear(10, 12), nn.Tanh(), nn.Linear(12, 8), nn.ReLU(), nn.Linear(8, 2))
        def train_torch(seed=42, epochs=350, lr=0.01, weight_decay=0.01):
            net_ = make_model(seed); opt = torch.optim.Adam(net_.parameters(), lr=lr, weight_decay=weight_decay); lossf = nn.MSELoss()
            xx, yy, xv, yv = (torch.tensor(a, dtype=torch.float32) for a in (Xf, Yf, Xv, Yv)); best, state, best_ep = float("inf"), None, 0
            for ep_ in range(1, epochs + 1):
                net_.train(); opt.zero_grad(); loss = lossf(net_(xx), yy); loss.backward(); opt.step(); net_.eval()
                with torch.no_grad(): vl = lossf(net_(xv), yv).item()
                if vl < best: best, best_ep, state = vl, ep_, {k: v.detach().clone() for k, v in net_.state_dict().items()}
            net_.load_state_dict(state); return net_, best, best_ep
        net_s, vbest, ep_s = train_torch()
        with torch.no_grad(): pr_t = net_s(torch.tensor(Xt, dtype=torch.float32)).numpy() * ysd + ymu
        m_t = mae(pr_t, yte); print(f"PyTorch, the tutorial's procedure (seed 42): selected epoch {ep_s}, minimum validation MSE {vbest:.5f} | test MAE dxy {m_t[0]:.4f}, gap {m_t[1]:.4f}")
        print("the tutorial's saved output:                  selected epoch 301, minimum validation MSE 0.08688 | test MAE dxy 0.0486, gap 0.0931")
        check("test MAE of the tutorial's training procedure within 0.01 eV of the tutorial's saved numbers (0.0486, 0.0931)", abs(m_t[0] - 0.0486) < 0.01 and abs(m_t[1] - 0.0931) < 0.015)
        check("... and the network does not beat Ridge on either target (Part I's finding)", m_t[0] > mae_ridge[0] and m_t[1] > mae_ridge[1] * 0.97)
        ms = []
        for s in range(10):
            n_, _, _ = train_torch(seed=s)
            with torch.no_grad(): ms.append(mae(n_(torch.tensor(Xt, dtype=torch.float32)).numpy() * ysd + ymu, yte))
        ms = np.array(ms); print(f"ten PyTorch seeds: test MAE dxy {ms[:, 0].mean():.4f} +- {ms[:, 0].std(ddof=1):.4f}, gap {ms[:, 1].mean():.4f} +- {ms[:, 1].std(ddof=1):.4f} | seeds below Ridge: dxy {(ms[:, 0] < mae_ridge[0]).sum()}, gap {(ms[:, 1] < mae_ridge[1]).sum()}")
    except Exception as e: print("PyTorch training stage could not be completed:", type(e).__name__, e)

In [ ]:
if HAVE_SK or HAVE_TORCH:
    ok = [lab for lab, passed in checks if passed]; bad = [lab for lab, passed in checks if not passed]
    print(f"Part II: {len(ok)} of {len(checks)} checks passed")
    for lab in bad: print("  not passed:", lab)

### What to look for
* **scikit-learn:** the split, the scaler and Ridge agree to rounding error: the baseline of Part I *is* the tutorial's baseline.
* **PyTorch, same weights:** the forward pass, the loss and all 254 gradients agree with autograd to float32 precision: the backpropagation of Part I is correct.
* **PyTorch, the tutorial's procedure:** numbers close to the tutorial's saved ones (they differ from the NumPy run of Part I because the random initializations differ), the network again not beating Ridge, and across ten seeds a spread like Part I's.